# 02. Feature Engineering

This notebook transforms raw transaction data into leakage-safe features for machine learning.

## Workflow

1. Load raw data
2. Convert date columns
3. Remove transactions occurring after the signal
4. Aggregate transactions by signal
5. Create statistical and behavioral features
6. Save processed datasets
7. Validate the resulting features

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

In [2]:
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

## 1. Load Raw Data

The dataset contains signal-level information and transaction-level information.

Transactions are linked to signals through `signal_id`.

In [3]:
train_signals = pd.read_csv(
    RAW_DIR / "train_signals.csv"
)

test_signals = pd.read_csv(
    RAW_DIR / "test_signals.csv"
)

train_transactions = pd.read_parquet(
    RAW_DIR / "train_transactions.parquet"
)

test_transactions = pd.read_parquet(
    RAW_DIR / "test_transactions.parquet"
)

print("Train signals:", train_signals.shape)
print("Test signals:", test_signals.shape)
print("Train transactions:", train_transactions.shape)
print("Test transactions:", test_transactions.shape)

Train signals: (14000, 3)
Test signals: (6000, 2)
Train transactions: (6987663, 5)
Test transactions: (3027575, 5)


## 2. Datetime Conversion

Convert signal and transaction timestamps to pandas datetime objects so that temporal relationships can be calculated.

In [4]:
train_signals["signal_sanasi"] = pd.to_datetime(
    train_signals["signal_sanasi"]
)

test_signals["signal_sanasi"] = pd.to_datetime(
    test_signals["signal_sanasi"]
)

train_transactions["tranzaksiya_vaqti"] = pd.to_datetime(
    train_transactions["tranzaksiya_vaqti"]
)

test_transactions["tranzaksiya_vaqti"] = pd.to_datetime(
    test_transactions["tranzaksiya_vaqti"]
)

## 3. Temporal Leakage Prevention

Only transactions that occurred on or before the signal date can be used.

Transactions occurring after the signal are removed because they contain information that would not have been available when the signal was generated.

In [5]:
train_tx = train_transactions.merge(
    train_signals[
        ["signal_id", "signal_sanasi", "eskalatsiya"]
    ],
    on="signal_id",
    how="left"
)

print("Before leakage filtering:", train_tx.shape)

Before leakage filtering: (6987663, 7)


In [6]:
future_train_tx = (
    train_tx["tranzaksiya_vaqti"]
    > train_tx["signal_sanasi"]
)

print(
    "Transactions after signal:",
    future_train_tx.sum()
)

Transactions after signal: 840


In [7]:
train_tx = train_tx[
    train_tx["tranzaksiya_vaqti"]
    <= train_tx["signal_sanasi"]
].copy()

print(
    "After leakage filtering:",
    train_tx.shape
)

After leakage filtering: (6986823, 7)


In [8]:
train_tx["days_before_signal"] = (
    train_tx["signal_sanasi"]
    - train_tx["tranzaksiya_vaqti"]
).dt.total_seconds() / (
    24 * 60 * 60
)

In [9]:
print(
    "Minimum days before signal:",
    train_tx["days_before_signal"].min()
)

print(
    "Maximum days before signal:",
    train_tx["days_before_signal"].max()
)

Minimum days before signal: 0.0
Maximum days before signal: 179.99997685185184


## 4. Prepare Test Transactions

The same temporal filtering logic is applied to the test transactions.

The test set does not contain the target variable.

In [10]:
test_tx = test_transactions.merge(
    test_signals[
        ["signal_id", "signal_sanasi"]
    ],
    on="signal_id",
    how="left"
)

print("Before leakage filtering:", test_tx.shape)

Before leakage filtering: (3027575, 6)


In [11]:
future_test_tx = (
    test_tx["tranzaksiya_vaqti"]
    > test_tx["signal_sanasi"]
)

print(
    "Transactions after signal:",
    future_test_tx.sum()
)

Transactions after signal: 51


In [12]:
test_tx = test_tx[
    test_tx["tranzaksiya_vaqti"]
    <= test_tx["signal_sanasi"]
].copy()

print(
    "After leakage filtering:",
    test_tx.shape
)

After leakage filtering: (3027524, 6)


In [13]:
test_tx["days_before_signal"] = (
    test_tx["signal_sanasi"]
    - test_tx["tranzaksiya_vaqti"]
).dt.total_seconds() / (
    24 * 60 * 60
)

## 5. Base Transaction Features

For each signal, calculate general transaction activity and amount statistics.

These features summarize the historical transaction behavior associated with each signal.

In [14]:
base_features_train = (
    train_tx
    .groupby("signal_id")
    .agg(
        transaction_count=(
            "signal_id",
            "size"
        ),
        miqdor_mean=(
            "miqdor_indeksi",
            "mean"
        ),
        miqdor_median=(
            "miqdor_indeksi",
            "median"
        ),
        miqdor_std=(
            "miqdor_indeksi",
            "std"
        ),
        miqdor_min=(
            "miqdor_indeksi",
            "min"
        ),
        miqdor_max=(
            "miqdor_indeksi",
            "max"
        )
    )
    .reset_index()
)

In [15]:
base_features_test = (
    test_tx
    .groupby("signal_id")
    .agg(
        transaction_count=(
            "signal_id",
            "size"
        ),
        miqdor_mean=(
            "miqdor_indeksi",
            "mean"
        ),
        miqdor_median=(
            "miqdor_indeksi",
            "median"
        ),
        miqdor_std=(
            "miqdor_indeksi",
            "std"
        ),
        miqdor_min=(
            "miqdor_indeksi",
            "min"
        ),
        miqdor_max=(
            "miqdor_indeksi",
            "max"
        )
    )
    .reset_index()
)

## 6. Transaction Direction Features

Calculate the number and proportion of incoming and outgoing transactions for each signal.

In [16]:
def add_direction_features(features, transactions):
    direction = (
        transactions
        .groupby(["signal_id", "kirim_chiqim"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    direction = direction.rename(
        columns={
            "kirim": "kirim_count",
            "chiqim": "chiqim_count"
        }
    )

    if "kirim_count" not in direction.columns:
        direction["kirim_count"] = 0

    if "chiqim_count" not in direction.columns:
        direction["chiqim_count"] = 0

    direction["kirim_ratio"] = (
        direction["kirim_count"]
        / (
            direction["kirim_count"]
            + direction["chiqim_count"]
        )
    )

    return features.merge(
        direction,
        on="signal_id",
        how="left"
    )

In [17]:
train_features = add_direction_features(
    base_features_train,
    train_tx
)

test_features = add_direction_features(
    base_features_test,
    test_tx
)

## 7. Transaction Type Features

Count transactions of each available transaction type for every signal.

In [18]:
def add_transaction_type_features(
    features,
    transactions
):
    type_counts = (
        transactions
        .groupby(["signal_id", "tranzaksiya_turi"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    type_counts = type_counts.rename(
        columns={
            "bank_otkazmasi": "bank_otkazmasi_count",
            "karta": "karta_count",
            "naqd": "naqd_count",
            "xalqaro": "xalqaro_count"
        }
    )

    expected_columns = [
        "bank_otkazmasi_count",
        "karta_count",
        "naqd_count",
        "xalqaro_count"
    ]

    for column in expected_columns:
        if column not in type_counts.columns:
            type_counts[column] = 0

    return features.merge(
        type_counts[
            ["signal_id"] + expected_columns
        ],
        on="signal_id",
        how="left"
    )

In [19]:
train_features = add_transaction_type_features(
    train_features,
    train_tx
)

test_features = add_transaction_type_features(
    test_features,
    test_tx
)

## 8. Recent Activity Features

Measure how many transactions occurred shortly before the signal.

Recent windows:
- 1 day
- 3 days
- 7 days
- 30 days

In [20]:
def add_recent_activity_features(
    features,
    transactions
):
    recent = (
        transactions
        .groupby("signal_id")
        .agg(
            tx_1d=(
                "days_before_signal",
                lambda x: (x <= 1).sum()
            ),
            tx_3d=(
                "days_before_signal",
                lambda x: (x <= 3).sum()
            ),
            tx_7d=(
                "days_before_signal",
                lambda x: (x <= 7).sum()
            ),
            tx_30d=(
                "days_before_signal",
                lambda x: (x <= 30).sum()
            )
        )
        .reset_index()
    )

    return features.merge(
        recent,
        on="signal_id",
        how="left"
    )

In [21]:
train_features = add_recent_activity_features(
    train_features,
    train_tx
)

test_features = add_recent_activity_features(
    test_features,
    test_tx
)

## 9. Amount Distribution Features

Capture the distribution of `miqdor_indeksi` using quantiles.

Quantiles can represent the shape of transaction amounts better than a mean alone.

In [22]:
def add_distribution_features(
    features,
    transactions
):
    distribution = (
        transactions
        .groupby("signal_id")
        .agg(
            miqdor_q10=(
                "miqdor_indeksi",
                lambda x: x.quantile(0.10)
            ),
            miqdor_q25=(
                "miqdor_indeksi",
                lambda x: x.quantile(0.25)
            ),
            miqdor_q75=(
                "miqdor_indeksi",
                lambda x: x.quantile(0.75)
            ),
            miqdor_q90=(
                "miqdor_indeksi",
                lambda x: x.quantile(0.90)
            )
        )
        .reset_index()
    )

    distribution["miqdor_iqr"] = (
        distribution["miqdor_q75"]
        - distribution["miqdor_q25"]
    )

    return features.merge(
        distribution,
        on="signal_id",
        how="left"
    )

In [23]:
train_features = add_distribution_features(
    train_features,
    train_tx
)

test_features = add_distribution_features(
    test_features,
    test_tx
)

## 10. Missing Value Handling

Some statistics, such as standard deviation, can be undefined for signals with a single transaction.

These values are replaced with zero.

In [24]:
train_features["miqdor_std"] = (
    train_features["miqdor_std"]
    .fillna(0)
)

test_features["miqdor_std"] = (
    test_features["miqdor_std"]
    .fillna(0)
)

In [25]:
train_features = train_features.merge(
    train_signals[
        ["signal_id", "signal_sanasi", "eskalatsiya"]
    ],
    on="signal_id",
    how="left"
)

test_features = test_features.merge(
    test_signals[
        ["signal_id", "signal_sanasi"]
    ],
    on="signal_id",
    how="left"
)

## 11. Processed Feature Dataset

The resulting datasets contain one row per signal.

In [26]:
print("Train features:", train_features.shape)
print("Test features:", test_features.shape)

print("\nTrain columns:")
print(train_features.columns.tolist())

print("\nTest columns:")
print(test_features.columns.tolist())

Train features: (14000, 25)
Test features: (6000, 24)

Train columns:
['signal_id', 'transaction_count', 'miqdor_mean', 'miqdor_median', 'miqdor_std', 'miqdor_min', 'miqdor_max', 'chiqim_count', 'kirim_count', 'kirim_ratio', 'bank_otkazmasi_count', 'karta_count', 'naqd_count', 'xalqaro_count', 'tx_1d', 'tx_3d', 'tx_7d', 'tx_30d', 'miqdor_q10', 'miqdor_q25', 'miqdor_q75', 'miqdor_q90', 'miqdor_iqr', 'signal_sanasi', 'eskalatsiya']

Test columns:
['signal_id', 'transaction_count', 'miqdor_mean', 'miqdor_median', 'miqdor_std', 'miqdor_min', 'miqdor_max', 'chiqim_count', 'kirim_count', 'kirim_ratio', 'bank_otkazmasi_count', 'karta_count', 'naqd_count', 'xalqaro_count', 'tx_1d', 'tx_3d', 'tx_7d', 'tx_30d', 'miqdor_q10', 'miqdor_q25', 'miqdor_q75', 'miqdor_q90', 'miqdor_iqr', 'signal_sanasi']


In [27]:
assert (
    train_features["signal_id"].nunique()
    == len(train_signals)
)

assert (
    test_features["signal_id"].nunique()
    == len(test_signals)
)

assert (
    set(train_features["signal_id"])
    == set(train_signals["signal_id"])
)

assert (
    set(test_features["signal_id"])
    == set(test_signals["signal_id"])
)

print("Signal coverage is valid.")

Signal coverage is valid.


In [28]:
print(
    "Train missing values:",
    train_features.isna().sum().sum()
)

print(
    "Test missing values:",
    test_features.isna().sum().sum()
)

Train missing values: 0
Test missing values: 0


In [29]:
feature_cols = [
    "transaction_count",
    "miqdor_mean",
    "miqdor_median",
    "miqdor_std",
    "miqdor_min",
    "miqdor_max",
    "chiqim_count",
    "kirim_count",
    "kirim_ratio",
    "bank_otkazmasi_count",
    "karta_count",
    "naqd_count",
    "xalqaro_count",
    "tx_1d",
    "tx_3d",
    "tx_7d",
    "tx_30d",
    "miqdor_q10",
    "miqdor_q25",
    "miqdor_q75",
    "miqdor_q90",
    "miqdor_iqr"
]

print("Number of ML features:", len(feature_cols))

Number of ML features: 22


In [30]:
assert all(
    column in train_features.columns
    for column in feature_cols
)

assert all(
    column in test_features.columns
    for column in feature_cols
)

assert train_features[feature_cols].isna().sum().sum() == 0
assert test_features[feature_cols].isna().sum().sum() == 0

print("Feature validation passed.")

Feature validation passed.


## 12. Save Processed Features

The final leakage-safe feature datasets are saved as Parquet files and used by the model training notebook.

In [31]:
train_features.to_csv(
    PROCESSED_DIR / "train_features.csv",
    index=False
)

test_features.to_csv(
    PROCESSED_DIR / "test_features.csv",
    index=False
)

print("Processed features saved.")

Processed features saved.


In [32]:
print("========== FEATURE ENGINEERING COMPLETE ==========")

print(
    f"Train signals: {len(train_features):,}"
)

print(
    f"Test signals: {len(test_features):,}"
)

print(
    f"ML features: {len(feature_cols)}"
)

print(
    f"Train shape: {train_features.shape}"
)

print(
    f"Test shape: {test_features.shape}"
)

print(
    f"Train file: {PROCESSED_DIR / 'train_features.csv'}"
)

print(
    f"Test file: {PROCESSED_DIR / 'test_features.csv'}"
)

========== FEATURE ENGINEERING COMPLETE ==========
Train signals: 14,000
Test signals: 6,000
ML features: 22
Train shape: (14000, 25)
Test shape: (6000, 24)
Train file: ../data/processed/train_features.csv
Test file: ../data/processed/test_features.csv
